In [22]:
using ModelingToolkitStandardLibrary.Electrical, ModelingToolkit, OrdinaryDiffEq, Test, Plots
using ModelingToolkit: t_nounits as t, D_nounits as D
using ModelingToolkitStandardLibrary.Blocks: Constant, Step, RealInput

In [23]:
@mtkmodel IdealSwitch begin
    @extend v, i = oneport = OnePort()

    @parameters begin
        state_init = false
        Gon = 1e5
        τ = 1e-3
    end

    @components begin
        input = RealInput()
    end

    @variables begin
        state(t)::Bool
        state_filtered(t) = Real(state_init)
        G(t)
    end

    @equations begin
        state ~ input.u
        D(state_filtered) ~ (state - state_filtered)/τ
        G ~ state_filtered*Gon
        i ~ G*v
    end
end;

In [24]:
@mtkmodel SwitchTest begin
    @parameters begin
        V = 10.0
        C = 1.0
        R = 1.0
    end

    @components begin
        step = Step(start_time=10.0, height=true, smooth=false)
        switch = IdealSwitch()
        voltage = Voltage()
        resistor = Resistor(R=R)
        capacitor = Capacitor(C=C, v=0.0)
        ground = Ground()
        constant = Constant(k=V)
    end

    @equations begin
        connect(voltage.p, switch.p)
        connect(switch.n, resistor.p)
        connect(resistor.n, capacitor.p)
        connect(voltage.n, capacitor.n, ground.g)
        connect(step.output, switch.input)
        voltage.V.u ~ V
    end
end;

In [25]:
@mtkbuild sys = SwitchTest()

println("System Equations:")
for (i, eq) in enumerate(full_equations(sys))
    println("$i. $eq")
end

MethodError: MethodError: no method matching -(::SymbolicUtils.BasicSymbolic{Real}, ::ODESystem)
The function `-` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  -(!Matched::ChainRulesCore.ZeroTangent, ::Any)
   @ ChainRulesCore C:\Users\matth\.julia\packages\ChainRulesCore\U6wNx\src\tangent_arithmetic.jl:101
  -(::Any, !Matched::ChainRulesCore.NoTangent)
   @ ChainRulesCore C:\Users\matth\.julia\packages\ChainRulesCore\U6wNx\src\tangent_arithmetic.jl:62
  -(!Matched::ChainRulesCore.NoTangent, ::Any)
   @ ChainRulesCore C:\Users\matth\.julia\packages\ChainRulesCore\U6wNx\src\tangent_arithmetic.jl:61
  ...


In [26]:
prob = ODEProblem(sys, [], (0.0, 25.0))
sol = solve(prob, Rodas4P());

ErrorException: A completed system is required. Call `complete` or `structural_simplify` on the system before creating an `ODEProblem`

In [27]:
plot(sol; vars = [sys.capacitor.v, sys.capacitor.i])

UndefVarError: UndefVarError: `sol` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [28]:
plot(sol; vars = [sys.switch.v, sys.switch.i])

UndefVarError: UndefVarError: `sol` not defined in `Main`
Suggestion: check for spelling errors or missing imports.